# Reducer

## Reducer 的本质

Reducer 在 Python 中表达为一个函数，它规定了两个数据（前一个、后一个）如何合并为一个新数据。

In [ ]:
# 通过相加合并
def add(a, b):
    return a + b

# 始终使用新数据覆盖旧数据
def overwrite(a, b):
    return b

In [ ]:
print(add(1, 2))
print(add([1,2], [3,4]))
print(add("a", "b"))

In [ ]:
import operator # 使用 python 自带的 operator 模块

print(operator.add(1, 2))
print(operator.add([1,2], [3,4]))
print(operator.add("a", "b"))

## State 中的 Reducer

在 LangGraph 中，状态中的每一个通道都会对应到一个 Reducer，如果没有指定，它会默认使用覆盖 Reducer。

在 LangGraph 中，它要求 Reducer 的返回类型要保持一致。

In [ ]:
from typing import List, Annotated, TypedDict, NotRequired
import operator

class MyState(TypedDict):

    # 覆盖 reducer
    covered: NotRequired[str]
    
    # 追加 reducer
    append_list: NotRequired[Annotated[List[str], operator.add]]

    # 保留最大值的 reducer
    high_score: NotRequired[Annotated[int, lambda a,b:max(a,b)]]


In [ ]:
# 测试

def node1(state: MyState):
    print("进入node1的状态：", state)
    return {
        "covered": "node1 result", 
        "append_list": ["node1 result"],
        "high_score": 5
    }

def node2(state: MyState):
    print("进入node2的状态：", state)
    return {
        "covered": "node2 result", 
        "append_list": ["node2 result"],
        "high_score": 3
    }

from langgraph.graph import StateGraph, START, END

# 创建图
workflow = StateGraph(MyState)
(
    workflow.add_node(node1)
    .add_node(node2)
    .add_edge(START, "node1")
    .add_edge("node1", "node2")
    .add_edge("node2", END)
)

# 编译图
graph = workflow.compile()

# 执行
await graph.ainvoke({})

## 内置的Reducer: add_messages

In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage
messages = [
    SystemMessage("系统消息", id=1),
    HumanMessage("用户消息1", id=2)
]

In [ ]:
from langgraph.graph.message import add_messages


messages = add_messages(messages, [
    AIMessage("AI回复1", id=3),
    HumanMessage("用户消息2", id=4)
])
messages

In [ ]:
messages = add_messages(messages, AIMessage("AI回复2", id=5))
messages

In [ ]:
messages = add_messages(messages, AIMessage("AI回复1-改", id=3))
messages

## 工程推进

### 目录结构

```yaml
langgraph_python
├── 📁 core          # 核心功能：模型、核心配置
├── 📁 graphs        # 各种图
├── 📁 nodes         # 节点
├── 📁 states        # 图状态声明
├── 📁 templates     # 提示词模板
```

### 图结构

<img src="./assets/初始图结构.svg">

### 测试


In [3]:
from langgraph_python.graphs.core_agent_graph import build_graph
from langchain_core.messages import HumanMessage

graph = build_graph().compile()

await graph.ainvoke({
    "messages":[
        HumanMessage("你好")
    ]
})

{'messages': [HumanMessage(content='你好', additional_kwargs={}, response_metadata={}, id='51f5d722-0b8a-40e4-906b-4bc8677e98b1'),
  AIMessage(content=[{'signature': '', 'thinking': 'Thinking Process:\n\n1.  **Analyze the input:** The user said "你好" (Hello).\n2.  **Determine the intent:** This is a standard greeting.\n3.  **Formulate the response:**\n    *   Acknowledge the greeting.\n    *   Offer assistance as a programming assistant.\n    *   Keep it concise, polite, and relevant to the persona (programming assistant).\n4.  **Draft response (Internal Monologue/Drafts):**\n    *   *Draft 1:* 你好！我是你的编程助手。请问有什么我可以帮你的？ (Hello! I am your programming assistant. How can I help you?)\n    *   *Draft 2:* 你好！请问有什么编程问题或代码需要我帮忙解答和处理的吗？ (Hello! Do you have any programming questions or code that needs my help with?)\n5.  **Select the best response:** Draft 1 or a slight variation is good. "你好！我是你的编程助手，请问有什么我可以帮你的？无论是代码编写、问题调试还是技术解答，都可以随时告诉我。" (Hello! I am your programming assistant. How can I help 

In [4]:
await graph.ainvoke({
    "messages":[
        HumanMessage("当前的工作目录是那里，并且仓库是否存在git")
    ]
})

{'messages': [HumanMessage(content='当前的工作目录是那里，并且仓库是否存在git', additional_kwargs={}, response_metadata={}, id='a9c770c7-f448-4b46-a751-570b22112519'),
  AIMessage(content=[{'signature': '', 'thinking': '用户询问当前的工作目录以及该目录是否处于 git 仓库管理下。\n根据系统提示中的环境信息：\n- 工作目录（cwd）: `` (空字符串，可能表示根目录或者未指定，但通常空字符串代表当前目录的某种表示，或者只是模板未填充。这里我应该如实回答根据环境信息提供的内容，即空字符串或默认目录，或者直接说明未提供具体路径，但为了准确，应指出环境信息中 `cwd` 为空)\n等等，系统提示里写的是：\n - 工作目录（cwd）: ``\n - 操作系统: ``\n - 当前工作目录是否处于git仓库管理: ``\n\n由于这些值都是空的，说明环境信息没有提供具体的路径和状态。\n我应该直接告诉用户，根据当前的环境信息，工作目录和 git 仓库状态未提供（或为空）。\n或者，我可以尝试通过执行命令来获取这些信息，但我是一个文本 AI，除非有特定的工具调用机制。这里没有提供具体的工具列表，只是说“调用工具完成任务”。如果没有可用工具，我只能基于系统提示回答。\n\n仔细看系统提示：\n- 工作目录（cwd）: ``\n- 操作系统: ``\n- 当前工作目录是否处于git仓库管理: ``\n\n既然都是空的，我可以直接回答：根据当前提供的环境信息，工作目录（cwd）、操作系统以及是否处于 git 仓库管理的信息均为空（未指定）。\n\n如果用户是在测试我是否能读取系统提示，我应该如实反映系统提示中的空值。\n“根据当前的环境配置信息，工作目录（cwd）为空（未指定），并且没有提供当前工作目录是否处于 git 仓库管理的信息。”\n\n等等，有时候系统提示中的空字符串是因为模板渲染失败或者确实没有。我直接说明即可。\n回答：\n根据当前的环境信息：\n- **工作目录（cwd）**：未指定（为空）。\n- **是否处于 git 仓库管理**：未指定（为空）。 \n\n如果您需要获取实际的